# GloGEM vs GloGEMflow: Full Alpine domain

Total volume and area timeseries for all Central European glaciers (RGI region 11),
comparing the dh/dt parametrisation against the SIA flowline model across three SSPs.

| Model | Geometry approach |
|-------|-------------------|
| **GloGEM dhdt** | dh/dt parametrisation (Huss et al. 2010) — solid lines |
| **GloGEMflow** | SIA flowline dynamics (Zekollari et al. 2019) — dashed lines |

Climate forcing: BCC-CSM2-MR, SSP1-2.6 / SSP2-4.5 / SSP5-8.5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

rcParams['font.family']       = 'sans-serif'
rcParams['font.sans-serif']   = ['Liberation Sans', 'Arial', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False
rcParams['axes.spines.top']    = False
rcParams['axes.spines.right']  = False

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust DIR_DHDT / DIR_FLOW to match dirres in your config.pro for each run.
# Adjust SUFFIX_DHDT / SUFFIX_FLOW to match your catchment_selection:
#   catchment_selection = ''          →  suffix = '_r1'
#   catchment_selection = 'my_name'   →  suffix = '_r1_my_name'
_BASE = '/scratch_net/vierzack04_fourth/jabeer/GloGEM/glogemflow_development'
_SUB  = 'monthly/CentralEurope/files/files_original'
_GCM  = 'BCC-CSM2-MR'

DIR_DHDT     = f'{_BASE}/alps_dhdt'
DIR_FLOW     = f'{_BASE}/alps_flow'
SUFFIX_DHDT  = '_r1'   # no catchment_selection → empty cc → suffix is just _r1
SUFFIX_FLOW  = '_r1'

# ── SSPs and styling ─────────────────────────────────────────────────────────
SSPS       = ['ssp126', 'ssp245', 'ssp585']
SSP_LABELS = {'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5', 'ssp585': 'SSP5-8.5'}
# IPCC AR6 standard colours
SSP_COLORS = {'ssp126': '#5492CD', 'ssp245': '#F6B334', 'ssp585': '#DC1C13'}
LW = 2.0

# ── Time axis ─────────────────────────────────────────────────────────────────
TRAN0   = 1940
N_YEARS = 161          # 1940–2100 inclusive
years   = np.arange(TRAN0, TRAN0 + N_YEARS)
I_2020  = 2020 - TRAN0  # index of 2020 (reference year for relative plots)

In [ ]:
def load_total(rundir, suffix, var, ssp, gcm=_GCM):
    """Sum `var` across all glaciers for one (rundir, ssp) combination.

    Returns an array of length N_YEARS, or None if the file is missing.
    The .dat file has RGI-ID as first column; subsequent columns are annual values
    starting at TRAN0.  We read the header to get the actual column count.
    """
    path = f'{rundir}/{_SUB}/{gcm}/{ssp}/centraleurope_{var}{suffix}.dat'
    try:
        with open(path) as fh:
            header = fh.readline().split()
            ncols  = len(header) - 1   # number of year columns
            total  = np.zeros(ncols)
            for line in fh:
                parts = line.split()
                if len(parts) < 2:
                    continue
                total += np.array(parts[1:ncols + 1], dtype=float)
    except FileNotFoundError:
        print(f'[missing] {path}')
        return None
    return total

In [ ]:
vol_dhdt  = {s: load_total(DIR_DHDT, SUFFIX_DHDT, 'Volume', s) for s in SSPS}
vol_flow  = {s: load_total(DIR_FLOW,  SUFFIX_FLOW,  'Volume', s) for s in SSPS}
area_dhdt = {s: load_total(DIR_DHDT, SUFFIX_DHDT, 'Area',   s) for s in SSPS}
area_flow = {s: load_total(DIR_FLOW,  SUFFIX_FLOW,  'Area',   s) for s in SSPS}

# Quick sanity check
for s in SSPS:
    v_d = vol_dhdt[s]
    v_f = vol_flow[s]
    tag = SSP_LABELS[s]
    if v_d is not None:
        print(f'{tag}  dhdt  vol_2020={v_d[I_2020]:.1f} km³   vol_2100={v_d[-1]:.1f} km³')
    if v_f is not None:
        print(f'{tag}  flow   vol_2020={v_f[I_2020]:.1f} km³   vol_2100={v_f[-1]:.1f} km³')

In [ ]:
# ── Figure 1: Total volume ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for ssp in SSPS:
    c   = SSP_COLORS[ssp]
    lbl = SSP_LABELS[ssp]
    if vol_dhdt[ssp] is not None:
        ax.plot(years, vol_dhdt[ssp], ls='-',  lw=LW, color=c)
    if vol_flow[ssp] is not None:
        ax.plot(years, vol_flow[ssp],  ls='--', lw=LW, color=c)

ax.axvline(2020, color='0.55', lw=0.9, ls=':', zorder=0)
ax.set_xlim(2000, 2100)
ax.set_ylim(bottom=0)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Total volume (km³)', fontsize=11)
ax.set_title('Central Europe — total glacier volume', fontsize=12)

legend_ssp = [Patch(color=SSP_COLORS[s], label=SSP_LABELS[s]) for s in SSPS]
legend_mdl = [
    Line2D([0], [0], color='0.3', ls='-',  lw=LW, label='dh/dt'),
    Line2D([0], [0], color='0.3', ls='--', lw=LW, label='GloGEMflow'),
]
ax.legend(handles=legend_ssp + legend_mdl, ncol=3, fontsize=9,
          loc='upper right', framealpha=0.85)
ax.grid(True, alpha=0.25, lw=0.6)

fig.tight_layout()
fig.savefig('../figs/alps_full_volume.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Volume relative to 2020 ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for ssp in SSPS:
    c = SSP_COLORS[ssp]
    if vol_dhdt[ssp] is not None and vol_dhdt[ssp][I_2020] > 0:
        ref = vol_dhdt[ssp][I_2020]
        ax.plot(years, vol_dhdt[ssp] / ref * 100, ls='-',  lw=LW, color=c)
    if vol_flow[ssp] is not None and vol_flow[ssp][I_2020] > 0:
        ref = vol_flow[ssp][I_2020]
        ax.plot(years, vol_flow[ssp]  / ref * 100, ls='--', lw=LW, color=c)

ax.axhline(100, color='0.55', lw=0.9, ls=':', zorder=0)
ax.axvline(2020, color='0.55', lw=0.9, ls=':', zorder=0)
ax.set_xlim(2000, 2100)
ax.set_ylim(bottom=0)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Volume relative to 2020 (%)', fontsize=11)
ax.set_title('Central Europe — relative volume (2020 = 100 %)', fontsize=12)

legend_ssp = [Patch(color=SSP_COLORS[s], label=SSP_LABELS[s]) for s in SSPS]
legend_mdl = [
    Line2D([0], [0], color='0.3', ls='-',  lw=LW, label='dh/dt'),
    Line2D([0], [0], color='0.3', ls='--', lw=LW, label='GloGEMflow'),
]
ax.legend(handles=legend_ssp + legend_mdl, ncol=3, fontsize=9,
          loc='upper right', framealpha=0.85)
ax.grid(True, alpha=0.25, lw=0.6)

fig.tight_layout()
fig.savefig('../figs/alps_full_volume_relative.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: Total area ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for ssp in SSPS:
    c = SSP_COLORS[ssp]
    if area_dhdt[ssp] is not None:
        ax.plot(years, area_dhdt[ssp], ls='-',  lw=LW, color=c)
    if area_flow[ssp] is not None:
        ax.plot(years, area_flow[ssp],  ls='--', lw=LW, color=c)

ax.axvline(2020, color='0.55', lw=0.9, ls=':', zorder=0)
ax.set_xlim(2000, 2100)
ax.set_ylim(bottom=0)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Total area (km²)', fontsize=11)
ax.set_title('Central Europe — total glacier area', fontsize=12)

legend_ssp = [Patch(color=SSP_COLORS[s], label=SSP_LABELS[s]) for s in SSPS]
legend_mdl = [
    Line2D([0], [0], color='0.3', ls='-',  lw=LW, label='dh/dt'),
    Line2D([0], [0], color='0.3', ls='--', lw=LW, label='GloGEMflow'),
]
ax.legend(handles=legend_ssp + legend_mdl, ncol=3, fontsize=9,
          loc='upper right', framealpha=0.85)
ax.grid(True, alpha=0.25, lw=0.6)

fig.tight_layout()
fig.savefig('../figs/alps_full_area.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
# Volume and area remaining in 2100 relative to 2020.
I_2100 = 2100 - TRAN0

header = f"{'SSP':<12}  {'V2020_d':>8}  {'V2100_d':>8}  {'Vloss_d':>8}  "
header +=           f"{'V2020_f':>8}  {'V2100_f':>8}  {'Vloss_f':>8}"
print(header)
print('-' * len(header))

for ssp in SSPS:
    lbl = SSP_LABELS[ssp]
    vd0 = vol_dhdt[ssp][I_2020] if vol_dhdt[ssp] is not None else float('nan')
    vd1 = vol_dhdt[ssp][I_2100] if vol_dhdt[ssp] is not None else float('nan')
    vf0 = vol_flow[ssp][I_2020]  if vol_flow[ssp]  is not None else float('nan')
    vf1 = vol_flow[ssp][I_2100]  if vol_flow[ssp]  is not None else float('nan')
    loss_d = (1 - vd1 / vd0) * 100 if vd0 > 0 else float('nan')
    loss_f = (1 - vf1 / vf0) * 100 if vf0 > 0 else float('nan')
    print(f'{lbl:<12}  {vd0:>8.1f}  {vd1:>8.1f}  {loss_d:>7.1f}%  '
          f'{vf0:>8.1f}  {vf1:>8.1f}  {loss_f:>7.1f}%')

print()
print('All values in km³.  V2020: volume in 2020.  Vloss: % volume loss 2020→2100.')